In [54]:
import pandas as pd
import torch
import os
import numpy as np
from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [55]:
#check we have access to the images
#gets image path sees if it exist
train_pairs = pd.read_csv("train_pairs.csv.gz")
first_img = train_pairs["image_path"].iloc[0]
print(first_img, ":", os.path.exists(first_img))

#open image and convert to pixels
#convert to 64 x 64
#make sure its RGB
img = Image.open(first_img)
img = img.convert("RGB")
img = img.resize((64, 64))
print(img.size)
print(img.mode)

#convert img to array
#convert to tensor array
#ensure tensor is in the right format
arr = np.array(img) / 255.0
tensor = torch.tensor(arr).permute(2, 0, 1).float()
print(tensor.shape, tensor.dtype)
print(tensor.min(), tensor.max())

abo-images-small/images/small/15/1533e482.jpg : True
(64, 64)
RGB
torch.Size([3, 64, 64]) torch.float32
tensor(0.) tensor(1.)


In [56]:
#encoder handles each tensor 
class encode_image(nn.Module):
    #the existing layers 
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(64 * 8 * 8, 128)
    #the order the layers work 
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.flatten(x)
        x = self.linear(x)
        return x

In [63]:
#fake batch(8) of images with random numbers 
#checks if encoder encodes to 8 images consisting of 128 numbers
encoder = encode_image()
fake = torch.randn(8, 3, 64, 64)
print(encoder(fake).shape)

torch.Size([8, 128])


In [64]:
#proper version of the logic 
#class inherits from Dataset
class ImgDataset(Dataset):
    #data stored in self.pairs
    def __init__(self, pairs):
        self.pairs = pairs
    #returns number of rows in data
    #so it knows how much it needs for shuffling and batching 
    def __len__(self):
        return len(self.pairs)
    #image loading
    def __getitem__(self, i):
        img_path = self.pairs["image_path"].iloc[i]
        label = self.pairs["label"].iloc[i]
        img = Image.open(img_path)
        img = img.convert("RGB")
        img = img.resize((64, 64))
        arr = np.array(img) / 255.0
        tensor = torch.tensor(arr).permute(2, 0, 1).float()
        return tensor, torch.tensor(label, dtype=torch.float32)

In [65]:
#loads and test it can read number of rows in train_pairs
#loads index 0 on train pairs 
dataset = ImgDataset(train_pairs)
print(len(dataset))
img, label = dataset[0]
print(img.shape, label)

167046
torch.Size([3, 64, 64]) tensor(0.)


In [66]:
#picks random 4 from dataset, then loads and encodes them
loader = DataLoader(dataset, batch_size=4, shuffle=True)
imgs, labels = next(iter(loader))
print(imgs.shape, labels.shape)
print(encoder(imgs).shape)

torch.Size([4, 3, 64, 64]) torch.Size([4])
torch.Size([4, 128])
